# Yolo - PreTrained

In [1]:
from ultralytics import YOLO
import os
import json
import shutil

In [3]:
!pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 7.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------- ----------------------------- 2.1/8.1 MB 10.5 MB/s eta 0:00:01
   ---------------------- ----------------- 4.5/8.1 MB 11.0 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.1 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 10.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------------------------- -- 2.1/2.2 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 9.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.6 MB 11.6 MB/s eta 0:00:01
   -------------- ------------

## Criar yaml para teste com o respetivo formato para Ultralytics YOLO 

Fazer yaml para os dados 
https://docs.ultralytics.com/pt/datasets/detect/#ultralytics-yolo-format

In [8]:
json_path = "annotations.json"
output_root = ""

# Criação de pastas
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(output_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_root, "labels", split), exist_ok=True)

# Carregar ficheiro JSON
with open(json_path, 'r') as f:
    data = json.load(f)

# Criar índice por id para acesso rápido
id_to_image = {img['id']: img for img in data['images']}

annotations = data['annotations']['pieces']
chess_board = data['annotations']['corners']

# Criação de índice de anotações por imagem_id
annotations_by_image = {}
for ann in annotations:
    image_id = ann['image_id']
    if ann.get('bbox'):
        bbox = ann['bbox']
    else:
        continue
    annotations_by_image.setdefault(image_id, []).append({
    "bbox": bbox,
    "category_id": ann['category_id']
})
    
# Criação de índice de anotações por imagem_id
chess_board_by_image = {}
for ann in chess_board:
    image_id = ann['image_id']
    if ann.get('corners'):
        corners = ann['corners']
    else:
        continue
    chess_board_by_image.setdefault(image_id, []).append({"corners": corners})

# Processar splits
for split in splits:
    ids = data['splits']['chessred2k'].get(split, [])
    for image_id in ids['image_ids']:
        image_info = id_to_image.get(image_id)
        if not image_info:
            print(f"[AVISO] ID {image_id} não encontrado em 'images'")
            continue

        src_path = "chessred2k/" + image_info.get('path')
        if not src_path or not os.path.isfile(src_path):
            print(f"[AVISO] Ficheiro não encontrado: {src_path}")
            continue

        dst_path = os.path.join(output_root, "Images", split, image_info['file_name'])
        shutil.copy2(src_path, dst_path)

        img_width = image_info['width']
        img_height = image_info['height']

        label_path = os.path.join(output_root, "Labels", split, image_info['file_name'].replace(".jpg", ".txt"))

        with open(label_path, 'w') as label_file:
            for items in chess_board_by_image.get(image_id, []):
                corners = items['corners']
                for corner in corners:
                    x, y = corners[corner]
                    w = 50
                    h = 50
                    # YOLO normalization
                    x_center_norm = x / img_width
                    y_center_norm = y / img_height
                    w_norm = w / img_width
                    h_norm = h / img_height
                    label_file.write(f"13 {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")
            for items in annotations_by_image.get(image_id, []):
                bbox = items['bbox'] 
                piece_categorie = items['category_id']
                x, y, w, h = bbox

                # Converter para formato YOLO
                x_center = (x + w / 2) / img_width
                y_center = (y + h / 2) / img_height
                w_norm = w / img_width
                h_norm = h / img_height

                label_file.write(f"{piece_categorie} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

## Train with a pre-trained model

In [ ]:
model = YOLO("yolo11n.yaml")  # build a new model from YAML
model = YOLO("yolo11n.pt")  # load a pretrained model
model = YOLO("yolo11n.yaml").load("yolo11n.pt")  # build from YAML and transfer weights

100%|██████████| 5.34M/5.34M [00:00<00:00, 6.74MB/s]


Transferred 691/691 items from pretrained weights


In [11]:
model.train(data="data.yaml", epochs=150, imgsz=640, patience=5)

New https://pypi.org/project/ultralytics/8.3.154 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.153  Python-3.13.3 torch-2.7.1+cpu CPU (Intel Core(TM) Ultra 7 165U)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train9, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

train: Scanning C:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\2st_part\labels\train... 1442 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1442/1442 [00:04<00:00, 345.72it/s]


train: New cache created: C:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\2st_part\labels\train.cache


c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


val: Fast image access  (ping: 0.20.1 ms, read: 105.49.9 MB/s, size: 1766.4 KB)


...

val: New cache created: C:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\2st_part\labels\val.cache



c:\Users\Dell\OneDrive - Confidencial Imobiliario\Documentos\ComputerVision\Task 3\1st_part\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Plotting labels to runs\detect\train9\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000556, momentum=0.9) with parameter groups 113 weight(decay=0.0), 120 weight(decay=0.0005), 119 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to runs\detect\train9
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/150         0G      1.616       4.78      1.139        462        640:   5%|▌         | 5/91 [00:40<11:32,  8.05s/it]


KeyboardInterrupt: 

In [ ]:
# Load a model
model = YOLO("path/to/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

# Teste the model
metrics = model.val(split="test")  
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

In [ ]:
# Predict with the model
results = model("image.jpg")  # predict on an image

# Access the results
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box